# Probabilistic Forecast + Prediction Intervals

**Week 7 · Lesson 8** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week7/lectures/08_probabilistic_forecast_prediction_intervals.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from fpppy Ch.5 (prediction intervals; conformal prediction) — puts ranges around Lesson 6's ARIMA forecaster and scores them with Lesson 7's tools. Data: Australian Bureau of Statistics, catalogue 8501.0 (CC BY 4.0).*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is
your first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** Beyond a single expected number, how much *range*
should a planner budget for next month's cafe/restaurant/takeaway
turnover — so staffing and stock cover the uncertainty, not just the
average case?

**Why it matters:** An interval that's too narrow means real surprises
happen more often than promised — understocking and understaffing at
exactly the moments a point forecast couldn't warn about. One that's too
wide wastes budget on padding nobody needed. So the interval has to
deliver the coverage it claims, and that gets *checked* on held-out
months, not assumed. Lesson 7 named the tools: an 80% interval is two
quantile forecasts (the 10th and 90th percentiles), built from past
misses, checked by coverage, and scored by pinball loss.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

The same series as Lesson 6: **Australian retail trade turnover**
(Australian Bureau of Statistics, catalogue 8501.0 table 11; CC BY 4.0),
national total for **"Cafes, restaurants and takeaway food services"**,
2004–2018, in $M AUD.

In [ ]:
from gbus564_coursework.utils.course_data import load_aus_retail

retail = load_aus_retail()
industry = retail[retail["Industry"] == "Cafes, restaurants and takeaway food services"]
series = industry.groupby("Month")["Turnover"].sum().loc["2004-01-01":"2018-12-01"]
series.index = pd.to_datetime(series.index)
series.describe()

### Data Prep

Intervals need **three** time blocks, not two:

- **Train** (Jan 2004 – Apr 2013) — fit the model.
- **Calibration** (May 2013 – Feb 2016, 34 months) — collect the model's
  one-month-ahead misses, which size the interval. Never used for fitting:
  the model's fit on its own training months has residuals that are partly
  memorized and would make it look more certain than it is. (A different
  use of the word from Lesson 7's calibration check on coverage: here it
  just names the held-out block whose misses size the interval.)
- **Test** (Mar 2016 – Dec 2018, 34 months) — the honest check of both the
  point forecast and the interval. The same test months as Lessons 6 and 7.

statsforecast wants the "long" table from Lesson 6: `unique_id`, `ds`
(date), `y` (value).

In [ ]:
CALIB_START = pd.Timestamp("2013-05-01")
TEST_START = pd.Timestamp("2016-03-01")
blocks = {"train": series.loc[:CALIB_START - pd.offsets.MonthBegin(1)],
          "calibration": series.loc[CALIB_START:TEST_START - pd.offsets.MonthBegin(1)],
          "test": series.loc[TEST_START:]}
for name, part in blocks.items():
    print(f"{name:12s} {part.index[0]:%b %Y} – {part.index[-1]:%b %Y}  ({len(part)} months)")

long = pd.DataFrame({"unique_id": "cafes", "ds": series.index, "y": series.values})

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = {"train": "#5F6368", "calibration": "#F9AB00", "test": "#202124"}
for name, part in blocks.items():
    ax.plot(part.index, part.values, color=colors[name], linewidth=1.4, label=name.capitalize())
ax.set_xlabel("Month"); ax.set_ylabel("Turnover ($M AUD)")
ax.set_title("Train, calibrate, test")
ax.legend(frameon=False)
plt.tight_layout()

## 3. Model

*Baseline first — the reference every model must beat — then the model
under study.*

**Baseline:** Lesson 3's **Seasonal Naive** — this month is the same month
last year.

**Model:** **ARIMA**, chosen automatically by statsforecast — one of the
two classical models that led Lesson 6's race, whose leftovers passed
Lesson 5's residual check. Same rules as Lesson 6: fit **once** on the
training months, then forecast **one month ahead** through the
calibration and test months, feeding in each new actual but never
refitting.

In [ ]:
%pip install -q statsforecast
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string

n_windows = len(blocks["calibration"]) + len(blocks["test"])
sf = StatsForecast(models=[AutoARIMA(season_length=12)], freq="MS")
# Despite the name, this isn't k-fold CV: with refit=False it fits ARIMA once on the
# training months, then rolls one month at a time through calibration and test,
# forecasting each month from the actuals before it (Lesson 2's rolling origin).
cv = sf.cross_validation(df=long, h=1, step_size=1, n_windows=n_windows,
                         refit=False, level=[80]).set_index("ds")

calib = cv.loc[:TEST_START - pd.offsets.MonthBegin(1)]
test = cv.loc[TEST_START:]
y_test = test["y"].values
pred_test = test["AutoARIMA"].values
seasonal_naive_test = series.shift(12).loc[TEST_START:].values

sf.fit(long[long["ds"] < CALIB_START])
print("ARIMA chose:", arima_string(sf.fitted_[0, 0].model_).strip())
print(f"Calibration forecasts: {len(calib)}, test forecasts: {len(test)}")

Fit on the shorter training window, the automatic search lands on a
slightly different form than Lesson 6's ARIMA(0,1,1)(0,1,1)[12] — an AR
and an MA term without the first difference — but the same seasonal
block: one seasonal difference and one seasonal MA term.

### Model Selection

Two ways to turn the same point forecast into an 80% interval:

- **The formula** — ARIMA's own interval, from its equations: estimate
  the spread of its errors and assume they're bell-shaped. It assumes the
  model is exactly right.
- **Past misses** (Lesson 7's recipe; the general idea behind *conformal
  prediction*) — look at how wrong the model actually was on the
  calibration months, and put that much room around each new forecast:
  the **10th and 90th percentiles of the signed misses** become the lower
  and upper bounds. No bell-curve assumption, and lopsided misses give a
  lopsided interval.

Why calibrate at all, when the formula comes free? In practice, formula
intervals tend to run narrow. Past misses are judged by how often the actual really landed
inside — and the recipe works on **any** model that makes a point
forecast, including tree ensembles and neural forecasters, which have no
formula to fall back on.

Look at the misses first.

In [ ]:
calib_misses = calib["y"] - calib["AutoARIMA"]          # actual minus forecast
q10, q90 = np.quantile(calib_misses, [0.1, 0.9])
print(f"Calibration misses: mean {calib_misses.mean():+.1f}, "
      f"10th percentile {q10:+.1f}, 90th percentile {q90:+.1f}  ($M)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.axvspan(q10, q90, color="#188038", alpha=0.18, label="Middle 80% of misses")
ax.hist(calib_misses, bins=15, color="#5F6368", edgecolor="white")
ax.axvline(0, color="#202124", linestyle=":", linewidth=1)
ax.set_xlabel("Calibration miss: actual − forecast ($M)"); ax.set_ylabel("Months")
ax.legend(frameon=False)
plt.tight_layout()

In [ ]:
# Each 80% interval is two quantile forecasts: a 10th and a 90th percentile.
# Both ARIMA intervals sit around the same point forecasts; only the width differs:
#   past misses -- computed here: the 10th/90th percentiles of the held-out
#                  calibration misses, added to each forecast (no bell-curve assumption)
#   formula     -- computed by statsforecast (level=[80]): the spread of ARIMA's
#                  in-sample training residuals, assumed bell-shaped
intervals = {
    "ARIMA, past misses": (pred_test + q10, pred_test + q90),
    "ARIMA, formula": (test["AutoARIMA-lo-80"].values, test["AutoARIMA-hi-80"].values),
}

# Baseline interval: Seasonal Naive, sized by ITS OWN calibration misses
sn_calib_misses = blocks["calibration"] - series.shift(12).loc[blocks["calibration"].index]
s10, s90 = np.quantile(sn_calib_misses, [0.1, 0.9])
intervals["Seasonal Naive, past misses"] = (seasonal_naive_test + s10, seasonal_naive_test + s90)

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

**Point forecast:** MAE against Seasonal Naive, as in Lesson 6, plus
Lesson 7's **MASE** — the same MAE divided by Seasonal Naive's typical
miss on the months before the test period, so a score below 1 beats
that yardstick. Both only score the center line; the interval needs its
own checks.

**Interval**, two checks from Lesson 7:

- **Coverage** — the interval's own promise. An 80% interval should
  contain the actual about 80% of the time on the test months.
- **Pinball loss** — the score for a quantile forecast, averaged over the
  10th and 90th percentiles. It rewards a band that's both honest and
  tight: a wide band can buy coverage, but pinball loss charges for the
  width. And it has a baseline: Seasonal Naive's interval, built the same
  way.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_pinball_loss

# The point forecast each interval is built around -- MAE scores that center line only
point_forecasts = {"ARIMA, past misses": pred_test, "ARIMA, formula": pred_test,
                   "Seasonal Naive, past misses": seasonal_naive_test}

# MASE scale (Lesson 7): Seasonal Naive's MAE on every month before the test period
pre_test = series.loc[:TEST_START - pd.offsets.MonthBegin(1)]
mase_scale = (pre_test - pre_test.shift(12)).abs().mean()
print(f"MASE scale: Seasonal Naive's typical pre-test miss, ${mase_scale:.0f}M")

rows = {}
for name, (lo, hi) in intervals.items():
    rows[name] = {
        "point MAE ($M)": mean_absolute_error(y_test, point_forecasts[name]),
        "point MASE": mean_absolute_error(y_test, point_forecasts[name]) / mase_scale,
        "avg width ($M)": np.mean(hi - lo),
        "coverage": np.mean((y_test >= lo) & (y_test <= hi)),
        "pinball loss": (mean_pinball_loss(y_test, lo, alpha=0.1)
                         + mean_pinball_loss(y_test, hi, alpha=0.9)) / 2,
    }
scores = pd.DataFrame(rows).T
print("80% intervals on the 34 test months:")
print(scores.round({"point MAE ($M)": 1, "point MASE": 2, "avg width ($M)": 0, "pinball loss": 1})
      .assign(coverage=scores["coverage"].map("{:.0%}".format)).to_string())

Both ARIMA intervals keep their promise — coverage in the low-to-mid 80s
against a claimed 80%, which is about as close as 34 months can show. The
past-misses band is a little tighter than the formula's for nearly the
same coverage, so it scores slightly better on pinball loss. Seasonal
Naive's interval also covers, but only by being about twice as wide —
which is exactly what pinball loss charges for: its score is close to
double ARIMA's.

The point-MAE column shows what pinball loss adds. The two ARIMA intervals
share the same center line, so their MAE is identical (\$43.1M, a MASE of
0.28 — the same score Lesson 7 reported) — yet
their ranges differ in width, and pinball loss tells them apart. MAE only
asks how close the center line lands; pinball loss asks how good the whole
range is, honest *and* tight. Here the gap between the two is small.
Seasonal Naive differs on both counts — a worse center line and a wider
band — and pinball loss charges for both.

One surprise against Lesson 7: there, the formula interval came out
*narrower* than the past-misses band, the textbook warning that formula
intervals tend to run narrow. Here it's the reverse, and the one-month band
is narrower than Lesson 7's too (about \$134M wide vs. \$165M), because these
calibration months (2013–16) were calmer than Lesson 7's 2010–15 window.
Which band comes out wider depends on the window the misses come from —
that's why coverage gets checked, not assumed.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

The interval, read as two quantile forecasts: the **lower edge is a 10th
percentile** ("only 1 month in 10 should come in below this") and the
**upper edge a 90th percentile** ("only 1 in 10 should come in above").

In [ ]:
lo80, hi80 = intervals["ARIMA, past misses"]
months = test.index

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.fill_between(months, lo80, hi80, color="#F9AB00", alpha=0.25,
                label="80% interval (from past misses)")
ax.plot(months, hi80, color="#F9AB00", linewidth=1, linestyle="--", label="90th percentile")
ax.plot(months, lo80, color="#F9AB00", linewidth=1, linestyle=":", label="10th percentile")
ax.plot(months, pred_test, color="#F9AB00", linewidth=2, label="ARIMA forecast")
ax.plot(months, y_test, color="#202124", linewidth=1.6, marker="o", markersize=3, label="Actual")
outside = (y_test < lo80) | (y_test > hi80)
ax.scatter(months[outside], y_test[outside], color="#D93025", zorder=5, s=40, label="Outside the band")
ax.set_xlabel("Month"); ax.set_ylabel("Turnover ($M AUD)")
ax.legend(frameon=False, fontsize=9, ncol=3)
plt.tight_layout()

In [ ]:
above = y_test > hi80     # actual above the band: would have run short
below = y_test < lo80     # actual below the band: would have overstocked
print(f"Outside the band: {above.sum() + below.sum()} of {len(y_test)} months "
      f"({above.sum()} above, {below.sum()} below)")
print("  above:", ", ".join(months[above].strftime("%b %Y")))
print("  below:", ", ".join(months[below].strftime("%b %Y")))
print(f"Margin around every forecast: {q10:+.0f} / {q90:+.0f} ($M)")

**Reading the band.** In planning terms, the interval says: *take ARIMA's
forecast, and expect next month to land within about \$67M either side of
it — and expect to be outside that range about one month in five.* On the
test months, that's close to what happened. The misses are split both
ways (more ran above the band, where staff and stock would have run short,
than below, where they'd have been left over), and no season piles up —
no sign that, say, every December breaks the band — though three of the
four above-band months came in 2016, early in the test period. A single
margin for every month looks reasonable for this series.

### What pinball loss rewards

Coverage and pinball loss react very differently to the band's *width*.
To see how, take ARIMA's band and squeeze or stretch it — the same center
line, with both margins multiplied by the same factor, from a quarter of
their size up to three times — and score every version on the test
months. (This is a diagnostic, not a way to pick the width: choosing it by
looking at the test months would be leakage.)

In [ ]:
scales = np.arange(0.25, 3.01, 0.05)
sweep = pd.DataFrame([{
    "width": np.mean((q90 - q10) * s),
    "coverage": np.mean((y_test >= pred_test + s * q10) & (y_test <= pred_test + s * q90)),
    "pinball": (mean_pinball_loss(y_test, pred_test + s * q10, alpha=0.1)
                + mean_pinball_loss(y_test, pred_test + s * q90, alpha=0.9)) / 2,
} for s in scales])
ours = scores.loc["ARIMA, past misses"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
ax1.plot(sweep["width"], sweep["coverage"], color="#1A73E8", linewidth=2)
ax1.axhline(0.8, color="#5F6368", linestyle=":", linewidth=1, label="Claimed 80%")
ax1.set_ylabel("Coverage on test months"); ax1.set_title("Coverage just keeps rising")
ax2.plot(sweep["width"], sweep["pinball"], color="#D93025", linewidth=2)
ax2.set_ylabel("Pinball loss (lower is better)"); ax2.set_title("Pinball loss bottoms out")
for ax, col in ((ax1, "coverage"), (ax2, "pinball loss")):
    ax.scatter(ours["avg width ($M)"], ours[col], color="#F9AB00", s=80, zorder=5,
               label="The calibrated band")
    ax.set_xlabel("Average band width ($M)")
    ax.legend(frameon=False)
plt.tight_layout()

best = sweep.loc[sweep["pinball"].idxmin()]
print(f"Lowest pinball loss: {best['pinball']:.1f} at width ${best['width']:.0f}M "
      f"(the calibrated band: {ours['pinball loss']:.1f} at ${ours['avg width ($M)']:.0f}M)")

The two curves pull in opposite directions. **Coverage only ever goes
up**: stretch the band far enough and it catches every month — 100% by
about \$200M wide — so coverage alone would reward padding without
limit. **Pinball loss is U-shaped.** Squeeze the band and the months that
land outside get charged at the steep rate, so the score climbs; stretch it
and every month pays a little more width fee, so the score climbs again.
The bottom is the tightest band that still catches about the share of
months it promised.

That's the precise sense in which pinball loss rewards bands that are
*honest and tight*: it rises with the range, so a narrower band that still
hits beats a wider one. And the past-misses recipe landed just beside the
bottom of the curve (8.1, against a best possible 8.0 at about \$121M)
without ever seeing the test months — the calibration misses sized the
band nearly as well as hindsight could have.

## 6. Recommend

**Recommendation:** Plan staffing and stock around ARIMA's interval, not
the point forecast alone. Built from the model's own past misses, the 80%
interval delivered close to its promised coverage on held-out months, at
about half the width of Seasonal Naive's — a trustworthy budget for the
uncertainty a single number can't show.

**The recipe travels.** Nothing in it was specific to ARIMA: a tree
ensemble, a neural network, or a foundation-model forecaster (Lesson 9)
can get a band the same way — any model that makes a point forecast,
including Lesson 6's ARIMA-plus-tree hybrid, which has no formula interval
at all: past misses are the only way to give it a range. The
model decides how *tight* the band can be, and coverage on held-out months
still decides whether to trust it.

**Which edge to plan around is a cost question, not a statistics one.**
Planning to the upper edge means planning to the **90th-percentile**
forecast — sensible only if running short costs about 9 times what
overstocking does (Lesson 7's pinball loss slides). If the real ratio is
different, the right percentile is different. The assignment makes that
decision with a stated cost ratio.

**Caveats / next steps:**

- **One month ahead only.** A single margin per percentile is right for
  next-month forecasts. A longer plan needs misses collected separately at
  each horizon, so the band can widen the way Lesson 7's fan chart did.
- **34 calibration months is a small sample** for estimating a 10th or
  90th percentile, and one test block is one test (Lesson 2). Refreshing
  the calibration window as new months arrive keeps the band honest.
- **Past misses only know the past.** A shock the calibration months never
  contained — a lockdown, a new competitor — isn't in the band. That's
  why coverage gets re-checked over time, not assumed.